<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">01 Data and Signals</h1>
<hr>


## **3. Fourier Approximation of a Rectangular Pulse**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)

</div>

### 3.1 Introduction

Notebook [01.2, Representing Signals in Time and Frequency Domain](01.2_Representing_Signals_in_Time_and_Frequency_Domain.ipynb), introduced the idea that sinusoids can combine to reproduce a waveform. We now explore that idea using a rectangular pulse.

A rectangular pulse is simple in the time domain but occupies an unlimited range of frequencies. This notebook develops that result in two complementary ways. First, we derive the continuous Fourier transform of an isolated rectangular pulse and examine how its duration controls the sinc-shaped spectrum. Then we sample the pulse, compute a Discrete Fourier Transform (DFT) with the Fast Fourier Transform (FFT), retain only a finite set of frequency components, and reconstruct a band-limited approximation.

Together, these views explain why shorter pulses require more bandwidth, why an ideal rectangle cannot pass unchanged through a finite-bandwidth system, and why truncated Fourier reconstructions oscillate near sharp transitions.

### 3.2 Continuous Fourier Transform: `rect` ↔ `sinc`

Let a unit-height rectangular pulse of duration $T>0$, centered at $t=0$, be

$$
x(t)=\operatorname{rect}\!\left(\frac{t}{T}\right)=
\begin{cases}
1, & |t|<T/2,\\
1/2, & |t|=T/2,\\
0, & |t|>T/2.
\end{cases}
$$

Using frequency $f$ in Hertz, its Fourier transform is

$$
\begin{aligned}
X(f)&=\int_{-\infty}^{\infty}x(t)e^{-j2\pi ft}\,dt\\
&=\int_{-T/2}^{T/2}e^{-j2\pi ft}\,dt\\
&=\frac{e^{-j\pi fT}-e^{j\pi fT}}{-j2\pi f}\\
&=T\,\frac{\sin(\pi fT)}{\pi fT}.
\end{aligned}
$$

The sine in the last step comes from **Euler's formula**. With $j^2=-1$,

$$
e^{j\theta}=\cos\theta+j\sin\theta,
\qquad
e^{-j\theta}=\cos\theta-j\sin\theta.
$$

Subtracting these expressions cancels the cosine terms:

$$
e^{-j\theta}-e^{j\theta}
=(\cos\theta-j\sin\theta)-(\cos\theta+j\sin\theta)
=-2j\sin\theta.
$$

Setting $\theta=\pi fT$ in the numerator therefore gives, for $f\ne0$,

$$
\frac{e^{-j\pi fT}-e^{j\pi fT}}{-j2\pi f}
=\frac{-2j\sin(\pi fT)}{-2j\pi f}
=\frac{\sin(\pi fT)}{\pi f}
=T\,\frac{\sin(\pi fT)}{\pi fT}.
$$

With the normalized definition $\operatorname{sinc}(u)=\sin(\pi u)/(\pi u)$,

$$
\boxed{X(f)=T\,\operatorname{sinc}(fT)}.
$$

At $f=0$, the expression is evaluated by continuity: $X(0)=T$. This is also the area under the pulse. Note that direct substitution into the intermediate expression containing $1/f$ gives the indeterminate form $0/0$, so its limiting value - or the original integral - must be used. The spectral zeros occur at $f=\pm 1/T,\pm 2/T,\ldots$.

### 3.3 What Does Bandwidth Mean Here?

The sinc spectrum has sidelobes that extend to arbitrarily high frequencies. Therefore, an ideal rectangular pulse is **not band-limited**: its absolute bandwidth is infinite. When a finite bandwidth is quoted, its definition must be stated. Common measures include:

- **First-null bandwidth:** $1/T$, measured from DC to the first positive zero.
- **Null-to-null main-lobe width:** $2/T$, measured between the first negative and positive zeros.
- **3 dB bandwidth:** the frequency interval over which the spectral power remains at least half its value at DC.

In this notebook, *main-lobe width* always means the null-to-null value $2/T$. It is not the absolute bandwidth. Thus, an ideal rectangular pulse has infinite absolute bandwidth even though its first-null bandwidth $1/T$ and null-to-null main-lobe width $2/T$ are finite. For example, when $T=0.4$ s, the first zeros occur at $f=\pm2.5$ Hz and the null-to-null main-lobe width is $5$ Hz. Note also the inverse dependence on pulse duration: reducing $T$ by a factor of two doubles both of these finite bandwidth measures, while increasing $T$ compresses the spectrum.

In [ ]:
from pathlib import Path
from runpy import run_path

_helper = next(root / ".admin/notebooks/fourier_approximation.py"
               for root in (Path.cwd(), *Path.cwd().parents)
               if (root / ".admin/notebooks/fourier_approximation.py").is_file())
_fourier = run_path(str(_helper))
_fourier["show_pulse_and_spectrum"]()


### 3.4 Finite-Bandwidth Fourier Approximation

An ideal rectangular pulse changes amplitude instantaneously at its edges. Producing such an abrupt transition would require frequency components extending without limit. Practical systems cannot respond infinitely fast: transmitters, communication channels, amplifiers, sensors, and other physical components operate over only a finite range of frequencies. This finite **bandwidth** limits how rapidly their output can change.

Consequently, a real system cannot reproduce an ideal rectangular pulse exactly. Frequencies outside the system's passband are strongly attenuated or removed, so the received or measured waveform is a band-limited approximation with rounded transitions and possible ringing. The experiment below models this by retaining only a limited number of Fourier components.

#### 3.4.1 From Samples to Frequency Components: DFT and FFT

The continuous Fourier transform describes a waveform $x(t)$ over continuous time. A computer instead works with a finite list of numbers. We obtain this list by **sampling** the waveform: taking its value at regularly spaced times. Let $N$ be the number of samples and $F_s$ the sampling rate in samples per second. The sample block represents an observation interval of duration $T_{\mathrm{obs}}=N/F_s$.

The **Discrete Fourier Transform (DFT)** represents this block of samples using a finite set of frequency components. Its output consists of complex numbers called **coefficients**, which contain magnitude and phase information. Each coefficient belongs to a **frequency bin**, meaning one frequency position on the discrete frequency grid. The **inverse DFT** adds the components back together. If all coefficients are retained, it recovers the original samples, apart from numerical rounding errors. Reproducing the continuous waveform between those samples is a separate question, explored in notebook 01.4.

The **Fast Fourier Transform (FFT)** is an efficient algorithm for calculating the DFT. It is not a different transform or a less accurate approximation. In this experiment, the approximation comes from deliberately discarding frequency components, not from using the FFT to calculate them.

The DFT treats the sample block as one period of a repeating signal. Its frequency bins are separated by

$$
\Delta f=\frac{F_s}{N}=\frac{1}{T_{\mathrm{obs}}}.
$$

Here, the observation interval is 10 s, so the bin spacing is 0.1 Hz. The calculation therefore represents a rectangular pulse repeated every 10 s, rather than a single isolated pulse extending over an unlimited time axis.

#### 3.4.2 Keeping Only a Limited Bandwidth

The slider selects the number $M$ of retained harmonics. We keep the **DC component**, which represents the average value, and the first $M$ harmonic pairs. Each pair contains the positive- and negative-frequency coefficients that together describe a real sinusoidal component. The highest retained positive frequency is the one-sided cutoff:

$$
f_{\mathrm{cutoff}}=M\Delta f.
$$

All coefficients beyond this cutoff are set to zero before the inverse transform reconstructs the waveform. With $\Delta f=0.1$ Hz, selecting $M=5$ gives a cutoff of 0.5 Hz, while $M=100$ gives a cutoff of 10 Hz. The right-hand plot shows the retained nonnegative-frequency components; the left-hand plot compares their reconstruction with the ideal rectangle.

#### 3.4.3 Why the Edges Ring: The Gibbs Phenomenon

Near each pulse edge, the reconstruction can rise above the top of the rectangle or dip below zero. These oscillations are called **ringing**. The characteristic overshoot near a jump when a Fourier representation is truncated is known as the **Gibbs phenomenon**.

A finite sum of smooth sinusoids cannot make an instantaneous jump. Near the edge, the retained components reinforce and cancel in a way that produces oscillations around the target levels. The missing higher-frequency components are needed to reproduce the sharp transition. This is a consequence of representing the jump with a limited frequency range, not noise in the data or a fault in the FFT.

Increase $M$ and watch the pulse edges. The transition becomes sharper, and the ringing becomes concentrated in a narrower region around each edge. For a Fourier approximation of a discontinuity, however, the peak overshoot does not simply disappear as more harmonics are added. Away from the edges, the approximation becomes closer to the target. At an ideal jump, the full Fourier representation converges to the average of the values on either side, here 0.5.

This time-frequency trade-off also matters in communication systems. If one pulse represents one symbol, increasing the symbol rate requires shorter pulses. Their spectra spread over a wider frequency range, so preserving the pulse shape requires more channel bandwidth.


In [ ]:
_fourier["show_approximation"]()


### 3.5 Digital Time Series as Rows of Rectangular Pulses

Sampling takes the value of a continuous-time signal $x(t)$ at regularly spaced times to form a sequence $x[n]=x(nT_s)$. The next notebook, [**Converting Analog Signals to Digital Time Series**](01.4_Converting_Analog_Signals_to_Digital_Time_Series.ipynb), examines this process in detail. The numbers $x[n]$ do not, by themselves, specify a continuous waveform between sampling instants. A common physical representation is the **zero-order hold (ZOH)**: each sample value is held constant for one sampling interval $T_s$. The resulting staircase signal is a weighted row of rectangular pulses:

$$
x_{\mathrm{ZOH}}(t)=\sum_n x[n]\,\operatorname{rect}\!\left(\frac{t-(n+\tfrac{1}{2})T_s}{T_s}\right).
$$

Every rectangle has duration $T_s$, begins at $nT_s$, and is scaled by the corresponding sample value $x[n]$. Thus, a zero-order hold creates a continuous staircase by holding each value $x[n]$ for one interval $T_s$. This interpretation connects a digital time series to the rectangular-pulse analysis developed in the preceding sections. Note that it is only one possible reconstruction rule; the abstract sequence $x[n]$ does not inherently specify a ZOH waveform.

If the complete signal is delayed by $t_0$, then $x_d(t)=x(t-t_0)$. The Fourier time-shift property gives

$$
X_d(f)=X(f)e^{-j2\pi ft_0}.
$$

Because $|e^{-j2\pi ft_0}|=1$, the delay leaves the magnitude spectrum unchanged: $|X_d(f)|=|X(f)|$. Consequently, every bandwidth definition based on spectral magnitude is unchanged. Only the phase changes, by the linear term $-2\pi ft_0$. It is important to distinguish a common shift from changes in relative timing. Shifting one isolated rectangle, or shifting the entire row of rectangles together, changes only spectral phase, while the magnitude spectrum and magnitude-based bandwidth remain unchanged. Moving individual rectangles independently can also change the magnitude spectrum through constructive and destructive interference.

In [ ]:
_fourier["show_zero_order_hold"]()


### 3.6 Key Takeaways

A unit-height rectangular pulse of duration $T$ has the continuous Fourier transform $X(f)=T\operatorname{sinc}(fT)$. Its first spectral zeros occur at $\pm1/T$, giving a null-to-null main-lobe width of $2/T$. The sinc sidelobes, however, extend to arbitrarily high frequencies. An ideal rectangular pulse therefore has infinite absolute bandwidth, even though finite measures such as first-null or 3 dB bandwidth can still be used to describe its spectral concentration. The inverse relationship between pulse duration and spectral width is fundamental: shorter pulses produce broader spectra, while longer pulses concentrate more of their energy near zero frequency.

For a finite sampled block, the DFT provides discrete frequency components separated by $1/T_{\mathrm{obs}}$ and implicitly treats the block as one period of a periodic signal. Reconstructing the pulse from only a limited number of harmonic pairs imposes a finite bandwidth. Retaining more harmonics reproduces finer time-domain detail and makes the transitions sharper, but any finite truncation leaves oscillations near the discontinuities because of the Gibbs phenomenon.

A zero-order hold connects this rectangular-pulse analysis to digital time series by representing each sample as a weighted rectangle of duration $T_s$. The values and relative timing of these rectangles determine the magnitude spectrum. Delaying the complete waveform multiplies its Fourier transform by a complex phase factor, changing the spectral phase but leaving the magnitude spectrum - and therefore every magnitude-based bandwidth measure - unchanged.

### 3.7 Literature / Background

1. Salih, M. S., & Salih, S. M. (2015). [**Fourier Transform: Signal Processing and Physical Sciences**](https://eth.swisscovery.slsp.ch/permalink/41SLSP_ETH/lshl64/alma99119130558105503). Rijeka, Croatia: IntechOpen.
2. [The Remarkable Story Behind The Most Important Algorithm Of All Time (YouTube)](https://www.youtube.com/watch?v=nmgFG7PUHfo&ab_channel=Veritasium)
3. [An Interactive Guide To The Fourier Transform](https://betterexplained.com/articles/an-interactive-guide-to-the-fourier-transform/)
4. [The Fourier Transform, Explained In One Sentence](https://blog.revolutionanalytics.com/2014/01/the-fourier-transform-explained-in-one-sentence.html)

---
### 3.8 Hands-On Exercise

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**For a pulse duration of $T=0.4$ s, where are the first spectral zeros, and what is the null-to-null main-lobe width?** Answer:

(add your answer here)

**If the pulse duration is reduced by a factor of two, what happens to the first-null bandwidth and the main-lobe width?** Answer:

(add your answer here)

**In the approximation experiment, what do $M$ and $\Delta f$ represent? What cutoff frequencies result from $M=5$ and $M=100$?** Answer:

(add your answer here)

**How does a zero-order hold construct a continuous-time waveform from the values $x[n]$? If the entire waveform is delayed by $t_0$, what changes in its spectrum and what remains unchanged?** Answer:

(add your answer here)

**If each rectangular pulse represents one data symbol, why does increasing the symbol rate generally require more channel bandwidth?** Answer:

(add your answer here)

**OPTIONAL: Are any explanations, tasks, or visualizations still unclear or in need of improvement?**

(please help the team by describing what was not understood)